In [4]:
from google.adk.agents import Agent
from google.adk.agents import LlmAgent
from google.adk.tools.google_search_tool import GoogleSearchTool

In [5]:
# Critique Agent - Reviews search results and suggests improvements
critique_agent = Agent(
    model="gemini-2.5-flash",
    name='critique_agent',
    description="Reviews responses and provides constructive feedback for improvement.",
    instruction="""You are a critique agent. You will receive a search response and must analyze it critically.

Your job is to:
1. Evaluate the completeness of the answer
2. Identify any missing information or unclear explanations
3. Suggest specific improvements

Output a structured critique with:
- STRENGTHS: What's good about the response
- WEAKNESSES: What's missing or could be better
- SUGGESTIONS: Specific recommendations for improvement

Be constructive and specific in your feedback.""",
)

# Refine Agent - Rewrites the response based on critique
refine_agent = Agent(
    model="gemini-2.5-flash",
    name='refine_agent',
    description="Rewrites and improves responses based on critique feedback.",
    instruction="""You are a refine agent. You will receive:
1. An original search response
2. A critique with suggestions for improvement

Your job is to rewrite the response incorporating the feedback from the critique.
Make the response:
- More complete and informative
- Clearer and better organized
- Address the weaknesses identified in the critique

Output ONLY the refined, improved response - no meta-commentary about your changes.""",
)

search_agent = LlmAgent(
    model="gemini-2.5-flash",  # A fast model suitable for search tasks
    name="SearchAgent",
    instruction="""You are an agent with the autonomy to research on the web.
    Search the requested topics using the 'google_search' tool and provide a summary of the findings.
    """,
    tools=[GoogleSearchTool()], # Add the built-in GoogleSearchTool
)

In [7]:
def append_to_state(tool_context, field, response):
  existing_state = tool_context.state.get(field, [])
  tool_context.state[field] = existing_state + [response]
  return {"status": "success"}

In [8]:
from google.adk.agents import LoopAgent

writers_room = LoopAgent(
  name="writers_room",
  description="Iterates ",
  sub_agents=[
  search_agent,
  critique_agent,
  refine_agent
  ],
  max_iterations=3,
)

In [9]:
root_agent = Agent(
  name="greeter",
  model="gemini-2.5-flash",
  description="You are helpful assistant.",
  instruction="""
    You are a helpful assistant that routes user requests to the appropriate specialist agent.
    For weather-related questions (forecasts, temperature, climate, etc.), delegate to weather_agent.
    For all other questions (news, general knowledge, searches), delegate to search_agent.
    Always delegate to the appropriate agent - do not try to answer directly.
  """,
  tools=[append_to_state],
  sub_agents=[writers_room],
)

In [10]:
from vertexai.preview import reasoning_engines
app = reasoning_engines.AdkApp(
  agent=root_agent,
  enable_tracing=False,
)

In [11]:
user_id = "test-user-id"
session = app.create_session(user_id=user_id)
print(session.id)

This legacy setting overrides the new Cloud Console toggle and environment variable controls.
Impact: The Cloud Console may incorrectly show telemetry as 'On' when it is actually 'Off', and the UI toggle will not work.
Action: To fix this and control telemetry, please remove the 'enable_tracing' parameter from your deployment code.
You can then use the 'GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY' environment variable:
agent_engines.create(
  env_vars={
    "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": true|false
  }
)
or the toggle in the Cloud Console: https://console.cloud.google.com/vertex-ai/agents.


a087ba2a-9411-4451-acfc-1e1f0ceba375


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:825: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


In [12]:
from IPython.display import Markdown, display
for event in app.stream_query(
  user_id=user_id,
  session_id=session.id,
  message="Explain Quantum computing",
):
  lastevent = event
display(Markdown(lastevent["content"]["parts"][0]["text"]))

Exception in thread Thread-8 (_asyncio_thread_main):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/adk/models/google_llm.py", line 241, in generate_content_async
    response = await self.api_client.aio.models.generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 7042, in generate_content
    response = await self._generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 5848, in _generate_content
    response = await self._api_client.async_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1432, in async_request
    result = await self._async_request(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_clien

KeyError: 'text'